# Import the Mount Isa network

### Objectives:
- Build an AequilibraE project from OpenStreetMap or Overture Maps data
- Construct a Bicycle route choice model
- Construct a custom utility function
- Explore differences between route choice set generation algorithms

In [ ]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import geopandas as gpd
import pandas as pd
from aequilibrae import Project

# A fresh folder lets the notebook be run more than once without replacing an existing project.
project_path = Path(gettempdir()) / f"aequilibrae_mount_isa_{uuid4().hex[:8]}"
print(f"AequilibraE project folder: {project_path}")


In [ ]:
import logging

from aequilibrae.utils import basic_config

logger = logging.getLogger("aequilibrae")
basic_config(logging.INFO)

In [ ]:
data_dir = Path.cwd() / "data"
assert data_dir.is_dir()

## Choose a network source

Use the bundled OpenStreetMap PBF for the workshop. It works offline and avoids overloading public servers when everyone runs the notebook at once.

Set `network_source` to `"overpass"` to download from OpenStreetMap via Overpass, or `"overture"` to download from Overture Maps. Both options need an internet connection. After changing the source, restart the kernel and run the notebook from the top to create a fresh project.

If you choose `"pbf"` and the Mount Isa file is missing, the next cell downloads the Queensland extract from Geofabrik and crops it to the workshop area.

In [ ]:
from shapely.geometry import box

network_source = "pbf"  # "pbf", "overpass", or "overture"
bbox = [139.45510, -20.79793, 139.57898, -20.60647]  # West, south, east, north
model_area = box(*bbox)

In [ ]:
osm_data_dir = data_dir / "osm"
pbf_path = osm_data_dir / "mount-isa.osm.pbf"

if network_source == "pbf":
    if not pbf_path.is_file():
        from urllib.request import urlretrieve

        from pyrosm import OSM

        osm_data_dir.mkdir(parents=True, exist_ok=True)
        geofabrik_url = "https://download.geofabrik.de/australia-oceania/australia/queensland-latest.osm.pbf"
        source_pbf_path = osm_data_dir / "queensland-latest.osm.pbf"
        print(f"Downloading {geofabrik_url}")
        urlretrieve(geofabrik_url, source_pbf_path)
        print(f"Cropping Queensland PBF to {bbox}")
        OSM(source_pbf_path, bounding_box=bbox).to_pbf(
            output_path=str(pbf_path), repack=True
        )

    print(f"Mount Isa PBF: {pbf_path}")

## Import the network into AequilibraE

All three sources use the same Mount Isa bounds and transport modes. The online sources may give different routes as their data changes.

In [ ]:
project = Project.create(project_path)

In [ ]:
modes = ("car", "transit", "bicycle", "walk")

if network_source == "pbf":
    project.tables.importer.osm(pbf_path=pbf_path, modes=modes, simplify=True)
elif network_source == "overpass":
    # To use a different Overpass server, uncomment these lines and replace the URL.
    # Use the server's base API URL. OSMnx adds /interpreter.
    # This changes the default for the current notebook kernel.
    # from aequilibrae.parameters import Parameters
    # Parameters._default["osm"]["overpass_endpoint"] = "https://overpass.example.org/api"
    project.tables.importer.osm(model_area=model_area, modes=modes, simplify=True)
elif network_source == "overture":
    project.tables.importer.overture(model_area=model_area, modes=modes, simplify=True)
else:
    raise ValueError('Choose "pbf", "overpass" or "overture" for network_source.')

## Inspect the imported network

The project keeps its network in the AequilibraE project database. Inspect the imported link and node tables and check the modes represented in the network.

In [ ]:
links = project.tables.links.data
nodes = project.tables.nodes.data
print(f"Links: {len(links):,}\nNodes: {len(nodes):,}")
links.head()

rename employment to labour

In [ ]:
zones_file = data_dir / "Mount Isa SA1.parquet"
employment_file = data_dir / "Mount Isa SA1 employment.csv"

In [ ]:
zones = gpd.read_parquet(zones_file)[["SA1_CODE_2021", "geometry"]].explode()
zones["SA1_CODE_2021"] = zones["SA1_CODE_2021"].astype(int)
zones = zones.rename(columns={"SA1_CODE_2021": "sa1_code_2021"}).set_index(
    "sa1_code_2021"
)
print(f"Zones: {len(zones):,}")
zones.head()

In [ ]:
employment = (
    pd.read_csv(employment_file, skiprows=9)  # Skip ABS header
    .tail(-1)  # Drop the first row which is blank for the two row header
    .head(-5)  # Drop the last 5 rows which contain data source info
    .loc[:, :"Total"]  # Drop columns after "Total", just whitespace
    .rename(
        columns={"1-digit level INDP Industry of Employment": "SA1_CODE_2021"}
    )  # Pandas reads this column name incorrectly due to the multi-row header
    .astype(int)
)

column_names = {
    column: column.lower().replace(" ", "_").replace(",", "")
    for column in employment.columns
} | {"SA1_CODE_2021": "sa1_code_2021"}

employment = employment.rename(columns=column_names).set_index("sa1_code_2021")

print(f"Employment: {len(employment):,}")
employment.head()

In [ ]:
zones = zones.join(employment, how="left")

Our dataframe has more columns than our database! Before we store our new zoning system we should create the columns we'd like to keep. This can be done through plan SQLite queries, or through the field editor.

In [ ]:
existing_columns = set(project.tables.zones.columns)
columns = set(zones.columns)

missing_columns = columns - existing_columns
print("Missing columns:", missing_columns)

In [ ]:
import sqlite3

try:
    with project.db_connection as conn:
        conn.execute('ALTER TABLE zones ADD COLUMN "SA1_CODE_2021" INTEGER')
except sqlite3.OperationalError as e:
    print("Error adding column:", e)

In [ ]:
fields = project.tables.zones.fields
existing_fields = fields.all_fields()
print("Existing zone fields:", existing_fields)

for proper_name, column in column_names.items():
    if column not in existing_fields:
        fields.add(column, description=f"ABS: {proper_name}", data_type="integer")
        logger.info("Added field: %s", column)

We'll table the zones dataframe that we've constructed and insert it into the database. Because we haven't supplied or set the ``zone_ids`` column ourselves, AequilibraE will infer that from the insertion order. If you need control over the IDs, say for instance, to match some existing IDs, then you should set that manually.

note about trying to prevent accidental double insertion

In [ ]:
zones["zone_id"] = project.tables.zones.insert_from(zones.reset_index())

Let's now add centroids to each of our zones, and connect them to the ``b`` mode  links with centroid connectors.

In [ ]:
project.tables.zones.add_centroids()
project.tables.zones.connect_mode("b")

## Explore the network

The height of each bar shows the zone’s 2021 Census population. Hover over a bar or zone to see the Census total and employment counts by industry for people who live there. Hover over a road to see its link ID and type.

In [ ]:
from route_choice_visualisation import network_overview

links = project.tables.links.data
nodes = project.tables.nodes.data
network_overview(links, nodes, zones, column_names)

## Simple example

In [ ]:
from aequilibrae.route_choice import RouteChoice

Here we'll build the road network from the project for a subset of modes. We're only interested in the ``b`` mode here.

In [ ]:
network = project.road.build_network(modes=["b"])

For now we'll use ``distance`` as utility.

You are encouraged to explore the construction of the utility function and it's effects on the ``probability`` column in the results.

In [ ]:
network.links["utility"] = network.links["distance"] * 0.1

We'll then build the graph ourselves to provide to ``RouteChoice``. We'll use the ``b`` mode and the ``distance`` field as the cost function.

In [ ]:
bicycle_graph = network.build_graph(
    mode="b",
    cost="utility",
    compact=True,
    remove_dead_ends=True,
)

For this small example we'll examine the route choice sets of the following OD pair.

In [ ]:
od_pairs = [(14, 37)]

We can then construct the ``RouteChoice`` object and set our parameters. Here we're using link penalisation with a maximum of 10 routs.

### Route choice parameters

#### Route cost cutoff

For each origin–destination pair, AequilibraE uses the lowest cost among the routes it found to set a cutoff for the choice model.

$$
\text{Cutoff cost} = \text{lowest route cost} \times \text{coefficient} + \text{constant}
$$

- `disutility_cutoff_coefficient` scales the lowest route cost. A value of `1.05` allows routes to cost up to 5% more when the constant is zero.
- `disutility_cutoff_constant` adds a fixed allowance in the same units as the route cost. In this example, cost is distance in metres.

With a coefficient of `1.05` and a constant of `0`, a lowest route cost of 4,000 metres gives a cutoff of 4,200 metres. Setting the constant to `100` raises the cutoff to 4,300 metres.

Routes at or below the cutoff are included in the probability calculation. Routes above it still appear in `choice.get_results()` with `mask=False` and a choice probability of zero.


#### Route search depth

With link penalisation, `max_depth` limits how many times AequilibraE searches for a route for each origin–destination pair. The first search finds the lowest-cost route using the original link costs. After each search, AequilibraE raises the cost of the links it used so that the next search is more likely to find a different route.

In this example, `max_depth=12` allows up to 12 searches, including the first one. `max_routes=10` stops the search once 10 distinct routes have been found. A search can find a route it has already seen, so AequilibraE may return fewer than 10 routes even after all 12 searches.

With BFSLE, `max_depth` limits how many levels of the search AequilibraE explores for each origin–destination pair. A level can contain several route searches.

- **Depth 0:** Find the lowest-cost route with all links available.
- **Depth 1:** Try excluding each link on that route in turn. Each exclusion gets its own search.
- **Depth 2 and beyond:** Keep the links already excluded in each branch of the search and exclude one more link from the route found in that branch.

In this example, `max_depth=12` lets BFSLE explore up to 12 levels, counting from depth 0. The search can stop earlier if it reaches `max_routes=10` or runs out of link combinations to try.

See [AequilibraE route choice documentation](https://www.aequilibrae.com/latest/python/route_choice/path_size_logit.html) for more information.

In [ ]:
choice = RouteChoice(bicycle_graph)
choice.set_choice_set_generation(
    algorithm="link-penalisation",
    max_routes=10,
    max_depth=12,
    disutility_cutoff_coefficient=1.05,
    disutility_cutoff_constant=0,
)

Because we're not providing a demand matrix here we'll call ``prepare`` with the set of OD pairs to compute paths for. If we'd like the path sized logit and probablity figures for the resulting routes we can set ``perform_assignment=True``, if we'd just like the routes themselves that can be set to ``False``.  

In [ ]:
choice.prepare(od_pairs)
choice.execute(perform_assignment=True)
route_results = choice.get_results()
route_results

In [ ]:
# We re-do everything with a sensitivity to distance 100X smaller and look at the results again
network.links["utility"] = network.links["distance"] * 0.001
bicycle_graph = network.build_graph(
    mode="b", cost="utility", compact=True, remove_dead_ends=True
)
choice = RouteChoice(bicycle_graph)
choice.set_choice_set_generation(
    algorithm="link-penalisation",
    max_routes=10,
    max_depth=12,
    disutility_cutoff_coefficient=1.05,
    disutility_cutoff_constant=0,
)

choice.prepare(od_pairs)
choice.execute(perform_assignment=True)
route_results = choice.get_results()
route_results

0/1 ODs processed                                 :   0%|          | 0/1 [00:00<?, ?it/s]

,origin id,destination id,cost,mask,path overlap,probability,route set,route directions
0,14,37,3.710712,True,0.212752,0.083534,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -..."
1,14,37,3.774021,True,0.327647,0.120754,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, -1..."
2,14,37,3.699323,True,0.193137,0.076701,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -..."
3,14,37,3.782134,True,0.370632,0.135493,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, 1,..."
4,14,37,3.773765,True,0.327601,0.120768,"[3812, 1984, 1837, 1831, 1834, 1826, 1825, 960...","[1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, 1, -1..."
5,14,37,3.691569,True,0.202234,0.080939,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, 1, -1..."
6,14,37,3.721922,True,0.281480,0.109287,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -..."
7,14,37,3.724980,True,0.342175,0.132447,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, 1, -1..."
8,14,37,3.688148,True,0.173418,0.069644,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, -1, -..."
9,14,37,3.687756,True,0.175308,0.070431,"[3812, 1984, 1837, 1831, 1827, 1822, 1812, 181...","[1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, 1, -1..."


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, PercentFormatter

route_groups = route_results.groupby(["origin id", "destination id"], sort=False)
route_labels, costs, probabilities = [], [], []
for (origin, destination), routes in route_groups:
    route_labels.append(f"{origin} --> {destination}")
    costs.append(routes["cost"].dropna())
    probabilities.append(routes["probability"].dropna())

with plt.style.context("seaborn-v0_8-whitegrid"):
    fig, (cost_ax, probability_ax) = plt.subplots(
        1, 2, figsize=(10, 4.3), sharey=True, layout="constrained"
    )
    fig.get_layout_engine().set(wspace=0.1)
    fig.suptitle("Route costs and choice probabilities", fontsize=16, fontweight="bold")

    cost_ax.hist(
        costs or [[]],
        bins="auto",
        stacked=True,
        label=route_labels or None,
        edgecolor="white",
        linewidth=0.8,
    )
    cost_ax.set(xlabel="Route cost", ylabel="Number of routes")
    cost_ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    probability_ax.hist(
        probabilities or [[]],
        bins=10,
        range=(0, 1),
        stacked=True,
        edgecolor="white",
        linewidth=0.8,
    )
    probability_ax.set(xlabel="Choice probability", xlim=(0, 1))
    probability_ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))

    for ax in (cost_ax, probability_ax):
        ax.set_axisbelow(True)
        ax.grid(False, axis="x")
        ax.grid(axis="y", color="#dde5ed")
        ax.spines[["top", "right"]].set_visible(False)

    if route_labels:
        fig.legend(
            loc="outside lower center", ncols=min(4, len(route_labels)), frameon=False
        )

    plt.show()

In [45]:
route_summary = route_groups[["cost", "probability"]].agg(["min", "median", "max"])
route_summary["cost"] = route_summary["cost"].round(2)
route_summary["probability"] *= 100
route_summary.insert(0, ("Routes", ""), route_groups.size())
route_summary = route_summary.rename(
    columns={
        "cost": "Route cost",
        "probability": "Choice probability (%)",
        "min": "Min",
        "median": "Median",
        "max": "Max",
    },
).rename_axis(index=["Origin", "Destination"])
route_summary.T

Origin                                14
Destination                           37
Routes                         10.000000
Route cost             Min      3.690000
                       Median   3.720000
                       Max      3.780000
Choice probability (%) Min      6.964400
                       Median   9.641091
                       Max     13.549280

## Extension: explore the differences in route choice set generation algorithms

In this advanced example we'll use the lonboard visulaisation library to create an interactive demo of the resulting routes.

### Build the bicycle routing graph

`BicycleRouteChoice` loads the bicycle network from the project, including its centroids and turn restrictions. It builds a compact graph using link distance in metres as the cost.

See [route_choice_workshop.py](route_choice_workshop.py) for the AequilibraE code. You can inspect the two result tables with `routing.compare((14, 37))` using just the routing module.

In [ ]:
from route_choice_workshop import BicycleRouteChoice

routing = BicycleRouteChoice(project)

Choose an origin and destination and click **Run comparison**. Both tabs use the same trip. BFSLE searches by removing links from routes it has found. LP searches again after increasing the cost of used links. Both use the same distance costs, route cap and 5% cutoff for inclusion. Each method applies the search depth limit differently.

- **BFSLE vs LP** shows the two choice sets with linked map views. Use **Compare link use** to highlight links shared by both methods and links found by just one, counting included routes in either direction.
- **Close a link** lets you click a bicycle road or enter its link ID. Click **Close link** to regenerate the choice set with that road closed in both directions. The closed link is red. **Reopen link** restores the original results.

Use **Route** to inspect an alternative, including excluded routes. The shortest route sits on top. Green marks the origin and purple marks the destination. Use **Fit zones** to see the whole zoning system. Hover inside a zone to see its centroid ID and SA1 code.

The maps and controls are in [route_choice_visualisation.py](route_choice_visualisation.py).

In [ ]:
from route_choice_visualisation import route_choice_workshop

# Refresh the map data after adding centroid connectors.
links = project.tables.links.data
nodes = project.tables.nodes.data

route_choice_workshop(routing, links, nodes, zones)